# Visualização: sífilis congênita em Porto Alegre

Este notebook cria uma visualização simples sobre desigualdade racial na sífilis congênita em Porto Alegre.

A ideia central é comparar a composição racial das mães em duas bases de 2024:

- SINASC: nascidos vivos em Porto Alegre, usado como contexto/denominador.
- SINAN/SIFCBR: notificações de sífilis congênita em residentes de Porto Alegre, usado como desfecho.

A visualização final pergunta: a participação de mães negras entre os casos notificados de sífilis congênita é maior do que entre os nascidos vivos do município?


## 1. Dependências

O Google Colab já inclui `pandas`, `matplotlib`, `seaborn` e `numpy`. Para este notebook, as únicas dependências adicionais são `pyreaddbc`, usada para converter arquivos `.dbc` do DATASUS para `.dbf`, e `dbfread`, usada para ler o `.dbf` convertido.

A célula abaixo só instala esses pacotes se eles ainda não estiverem disponíveis. Ela não reinstala bibliotecas nativas do Colab e não reinicia o runtime automaticamente.


In [ ]:
import importlib.util
import subprocess
import sys

pacotes = []
if importlib.util.find_spec("pyreaddbc") is None:
    pacotes.append("pyreaddbc==2.0.4")
if importlib.util.find_spec("dbfread") is None:
    pacotes.append("dbfread==2.0.7")

if pacotes:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", *pacotes])
    print("Pacotes instalados:", ", ".join(pacotes))
else:
    print("Dependências já instaladas.")


## 2. Bibliotecas e configurações

Esta seção usa as bibliotecas já disponíveis no Colab para manipulação e visualização dos dados.


In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

CODIGO_PORTO_ALEGRE = "431490"


## 3. Caminhos dos dados

A estrutura esperada no repositório é:

- `data/raw/SIFCBR24.dbc`
- `data/raw/sinasc/DNRS2024.dbc`
- `data/raw/cnes/STRS24*.dbc`


In [ ]:
def encontrar_arquivo(*candidatos):
    for candidato in candidatos:
        caminho = Path(candidato)
        if caminho.exists():
            return caminho
    caminhos = ", ".join(str(c) for c in candidatos)
    raise FileNotFoundError(f"Nenhum arquivo encontrado. Caminhos testados: {caminhos}")

ARQUIVO_SINAN = encontrar_arquivo(
    "../data/raw/SIFCBR24.dbc",
    "data/raw/SIFCBR24.dbc",
    "SIFCBR24.dbc",
)

ARQUIVO_SINASC = encontrar_arquivo(
    "../data/raw/sinasc/DNRS2024.dbc",
    "data/raw/sinasc/DNRS2024.dbc",
    "DNRS2024.dbc",
)

print("SINAN:", ARQUIVO_SINAN)
print("SINASC:", ARQUIVO_SINASC)


## 4. Leitura dos arquivos `.dbc`

O pacote `pyreaddbc` converte os arquivos comprimidos `.dbc` do DATASUS para `.dbf`. Em seguida, `dbfread` lê o `.dbf` convertido e o resultado é transformado em um DataFrame pandas.


In [ ]:
import tempfile
from pathlib import Path

import pyreaddbc
from dbfread import DBF, FieldParser


class ParserDatasInvalidas(FieldParser):
    def parseD(self, field, data):
        try:
            return super().parseD(field, data)
        except ValueError:
            return None


def ler_dbc(caminho_dbc, encoding="iso-8859-1"):
    caminho_dbc = Path(caminho_dbc)
    with tempfile.TemporaryDirectory() as tmpdir:
        caminho_dbf = Path(tmpdir) / f"{caminho_dbc.stem}.dbf"
        pyreaddbc.dbc2dbf(str(caminho_dbc), str(caminho_dbf))
        tabela = DBF(
            str(caminho_dbf),
            encoding=encoding,
            parserclass=ParserDatasInvalidas,
            load=True,
        )
        return pd.DataFrame(iter(tabela))


sinan = ler_dbc(ARQUIVO_SINAN)
sinasc = ler_dbc(ARQUIVO_SINASC)

print("SINAN", sinan.shape)
print("SINASC", sinasc.shape)


## 5. Recorte: Porto Alegre

Nos microdados, o código de Porto Alegre aparece como `431490` ou com dígito adicional. Por isso o filtro usa `startswith("431490")`.


In [ ]:
sinan_poa = sinan[
    sinan["ID_MN_RESI"].astype(str).str.strip().str.startswith(CODIGO_PORTO_ALEGRE)
].copy()

sinasc_poa = sinasc[
    sinasc["CODMUNRES"].astype(str).str.strip().str.startswith(CODIGO_PORTO_ALEGRE)
].copy()

print("Casos SINAN Porto Alegre:", len(sinan_poa))
print("Nascidos vivos SINASC Porto Alegre:", len(sinasc_poa))


## 6. Preparação dos indicadores cruzados

A análise abaixo cruza raça/cor materna com marcadores de acesso e vulnerabilidade social. Para facilitar a comparação visual, o recorte agrupa mães pretas, pardas e indígenas em um único bloco analítico: `Mães negras e indígenas`. Esse agrupamento destaca desigualdades estruturais sem fragmentar demais os grupos com poucos registros.

Indicadores usados:

- SINAN: proporção de casos de sífilis congênita em que a mãe não realizou pré-natal.
- SINASC: proporção de nascidos vivos com menos de 7 consultas de pré-natal.
- SINASC: proporção de nascidos vivos de mães com baixa escolaridade, aproximada por até 7 anos de estudo no campo `ESCMAE`.

Cada barra mostra percentual e contagem no formato `n/total`.

In [ ]:
def grupo_raca_comparativo(valor):
    codigo = str(valor).strip()
    if codigo == "1":
        return "Mães brancas"
    if codigo in {"2", "4", "5"}:
        return "Mães negras e indígenas"
    return "Outras/sem informação"


def resumo_percentual(df, grupo_col, flag_col, indicador, fonte, ordem_grupos):
    base = df[df[grupo_col].isin(ordem_grupos)].copy()
    if base.empty:
        return pd.DataFrame(columns=["grupo_raca", "indicador", "fonte", "percentual", "numerador", "denominador"])

    resumo = (
        base
        .groupby(grupo_col)
        .agg(
            numerador=(flag_col, "sum"),
            denominador=(flag_col, "size"),
        )
        .reset_index()
        .rename(columns={grupo_col: "grupo_raca"})
    )
    resumo["percentual"] = resumo["numerador"] / resumo["denominador"] * 100
    resumo["indicador"] = indicador
    resumo["fonte"] = fonte
    return resumo[["grupo_raca", "indicador", "fonte", "percentual", "numerador", "denominador"]]


ordem_grupos = ["Mães brancas", "Mães negras e indígenas"]
ordem_indicadores = [
    "Casos sem pré-natal",
    "Nascidos com <7 consultas",
    "Baixa escolaridade materna",
]

sinan_ind = sinan_poa.copy()
sinan_ind["grupo_raca"] = sinan_ind["ANT_RACA"].map(grupo_raca_comparativo)
sinan_ind["pre_natal"] = sinan_ind["ANT_PRE_NA"].astype(str).str.strip()
sinan_ind = sinan_ind[sinan_ind["pre_natal"].isin(["1", "2"])]
sinan_ind["sem_pre_natal"] = sinan_ind["pre_natal"].eq("2")

sinasc_ind = sinasc_poa.copy()
sinasc_ind["grupo_raca"] = sinasc_ind["RACACORMAE"].map(grupo_raca_comparativo)
sinasc_ind["consultas_pre_natal"] = pd.to_numeric(sinasc_ind["CONSPRENAT"], errors="coerce")
sinasc_ind["escolaridade"] = sinasc_ind["ESCMAE"].astype(str).str.strip()

sinasc_prenatal = sinasc_ind[
    sinasc_ind["consultas_pre_natal"].notna()
    & sinasc_ind["consultas_pre_natal"].between(0, 98)
].copy()
sinasc_prenatal["menos_de_7_consultas"] = sinasc_prenatal["consultas_pre_natal"].lt(7)

sinasc_escolaridade = sinasc_ind[sinasc_ind["escolaridade"].isin(["1", "2", "3", "4", "5"])].copy()
sinasc_escolaridade["baixa_escolaridade"] = sinasc_escolaridade["escolaridade"].isin(["1", "2", "3"])

indicadores = pd.concat(
    [
        resumo_percentual(
            sinan_ind,
            "grupo_raca",
            "sem_pre_natal",
            "Casos sem pré-natal",
            "SINAN/SIFCBR",
            ordem_grupos,
        ),
        resumo_percentual(
            sinasc_prenatal,
            "grupo_raca",
            "menos_de_7_consultas",
            "Nascidos com <7 consultas",
            "SINASC",
            ordem_grupos,
        ),
        resumo_percentual(
            sinasc_escolaridade,
            "grupo_raca",
            "baixa_escolaridade",
            "Baixa escolaridade materna",
            "SINASC",
            ordem_grupos,
        ),
    ],
    ignore_index=True,
)

indicadores["grupo_raca"] = pd.Categorical(indicadores["grupo_raca"], categories=ordem_grupos, ordered=True)
indicadores["indicador"] = pd.Categorical(indicadores["indicador"], categories=ordem_indicadores, ordered=True)
indicadores = indicadores.sort_values(["indicador", "grupo_raca"])
indicadores

## 7. Tabela auxiliar para interpretação

A tabela abaixo mostra numerador, denominador e percentual de cada indicador. Ela ajuda a evitar conclusões fortes quando algum grupo tem poucos registros.

In [ ]:
tabela_indicadores = indicadores.copy()
tabela_indicadores["percentual"] = tabela_indicadores["percentual"].round(1)
tabela_indicadores["n"] = (
    tabela_indicadores["numerador"].astype(int).astype(str)
    + "/"
    + tabela_indicadores["denominador"].astype(int).astype(str)
)

tabela_indicadores[["grupo_raca", "fonte", "indicador", "n", "percentual"]]

## 8. Visualização final

A figura usa barras agrupadas para facilitar a comparação direta entre `Mães brancas` e `Mães negras e indígenas`. Cada rótulo mostra o percentual e a contagem usada no cálculo.

In [ ]:
plot_data = indicadores.copy()
plot_data["rotulo"] = plot_data.apply(
    lambda row: f"{row.percentual:.1f}%\n({int(row.numerador)}/{int(row.denominador)})",
    axis=1,
)

fig, ax = plt.subplots(figsize=(11, 6))

paleta = {
    "Mães brancas": "#4C78A8",
    "Mães negras e indígenas": "#B55A30",
}

sns.barplot(
    data=plot_data,
    x="indicador",
    y="percentual",
    hue="grupo_raca",
    order=ordem_indicadores,
    hue_order=ordem_grupos,
    palette=paleta,
    ax=ax,
)

for container in ax.containers:
    labels = []
    for bar in container:
        altura = bar.get_height()
        labels.append("" if pd.isna(altura) else f"{altura:.1f}%")
    ax.bar_label(container, labels=labels, padding=3, fontsize=9)

for row in plot_data.itertuples(index=False):
    x_base = ordem_indicadores.index(row.indicador)
    deslocamento = -0.2 if row.grupo_raca == ordem_grupos[0] else 0.2
    ax.text(
        x_base + deslocamento,
        max(row.percentual - 2.5, 0.5),
        f"{int(row.numerador)}/{int(row.denominador)}",
        ha="center",
        va="top",
        fontsize=8,
        color="white" if row.percentual >= 8 else "#333333",
        weight="bold" if row.percentual >= 8 else "normal",
    )

ax.set_title(
    "Sífilis congênita em Porto Alegre (2024): acesso e vulnerabilidade por raça/cor materna",
    fontsize=14,
    weight="bold",
    pad=16,
)
ax.set_xlabel("")
ax.set_ylabel("Percentual dentro de cada grupo")
ax.set_ylim(0, max(40, plot_data["percentual"].max() + 8))
ax.legend(title="Grupo materno", loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.25)
ax.grid(axis="x", visible=False)
ax.set_xticklabels(ax.get_xticklabels(), rotation=12, ha="right")

plt.tight_layout()

saida_dir = Path("../outputs/figures")
if not saida_dir.exists():
    saida_dir = Path("outputs/figures")
saida_dir.mkdir(parents=True, exist_ok=True)

plt.savefig(saida_dir / "visualizacao_sifilis_congenita_poars.png", dpi=200, bbox_inches="tight")
plt.show()

## 9. Como usar no relatório

Imagem gerada: `outputs/figures/visualizacao_sifilis_congenita_poars.png`.

Caption sugerido para você adaptar com suas palavras: a figura compara mães brancas com mães negras e indígenas em três marcadores associados à sífilis congênita e à vulnerabilidade social em Porto Alegre em 2024. As barras mostram percentuais dentro de cada grupo, e os rótulos internos apresentam a contagem `n/total`. O agrupamento de mães pretas, pardas e indígenas facilita a leitura visual das desigualdades sem esconder os denominadores usados no cálculo.